In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import joblib

In [59]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/CKD Project/kidney_disease.csv")
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


**Impute Missing Values Using MICE**

In [60]:
CAT_COLS = ['rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']

df_encoded = df.copy()
binary_map = {'yes':1,'no':0,'normal':1,'abnormal':0, 'present':1,'notpresent':0,'good':1,'poor':0}
for col in CAT_COLS:
    df_encoded[col] = df_encoded[col].map(binary_map)
df_encoded['class'] = df_encoded['class'].map({'ckd':1, 'notckd':0})
df_encoded.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,1.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,NaN,1.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,1.0,1.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,1.0,0.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,1.0,1.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0,1


In [61]:
target_col  = df_encoded['class']
df_encoded.drop(columns=['class'], inplace=True)

In [62]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns.tolist())

In [63]:
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
df_imputed = pd.DataFrame(knn_imputer.fit_transform(df_scaled), columns=df_encoded.columns.tolist())

In [64]:
# ── 7. Inverse scale to get real values back ──
df_imputed = pd.DataFrame(scaler.inverse_transform(df_imputed), columns=df_encoded.columns.tolist())

In [65]:
# Correct dtypes
INT_COLS   = ['age','bp','al','su','bgr','pcv','wbcc','sod']
FLOAT_COLS = ['sg','bu','sc','pot','hemo','rbcc']

# Clip categoricals to 0/1
for col in CAT_COLS:
    df_imputed[col] = df_imputed[col].clip(0, 1).round().astype(int)
for col in INT_COLS:
    df_imputed[col] = df_imputed[col].round().astype(int)
for col in FLOAT_COLS:
    df_imputed[col] = df_imputed[col].round(3)


In [66]:
df_imputed.sample(50)

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
231,60,90,1.015,2,2,1,1,0,0,269,...,11.500,35,9632,3.994,1,1,1,1,1,0
207,50,70,1.010,0,0,1,1,0,0,230,...,12.000,41,10400,4.600,1,1,0,1,0,0
297,53,60,1.025,0,0,1,1,0,0,116,...,15.800,45,7700,5.200,0,0,0,1,0,0
120,72,90,1.025,1,3,0,1,0,0,323,...,12.600,36,10063,3.826,0,1,1,0,0,0
310,46,60,1.020,0,0,1,1,0,0,102,...,13.200,44,11000,5.400,0,0,0,1,0,0
144,60,90,1.010,2,0,0,1,0,0,105,...,11.100,33,10500,4.100,0,0,0,1,0,0
388,51,80,1.020,0,0,1,1,0,0,94,...,15.500,46,9500,6.400,0,0,0,1,0,0
106,50,90,1.012,2,0,0,1,0,0,89,...,6.000,17,6500,3.145,1,1,0,1,1,1
58,73,80,1.020,2,0,0,0,0,0,253,...,10.500,33,7200,4.300,1,1,1,1,0,0
195,70,90,1.020,2,1,0,0,0,1,184,...,5.800,27,7786,3.456,1,1,1,0,0,0


In [67]:
df_imputed.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
0,48,80,1.020,1,0,1,1,0,0,121,...,15.4,44,7800,5.200,1,1,0,1,0,0
1,7,50,1.020,4,0,1,1,0,0,96,...,11.3,38,6000,5.334,0,0,0,1,0,0
2,62,80,1.010,2,3,1,1,0,0,423,...,9.6,31,7500,3.829,0,1,0,0,0,1
3,48,70,1.005,4,0,1,0,1,0,117,...,11.2,32,6700,3.900,1,0,0,0,1,1
4,51,80,1.010,2,0,1,1,0,0,106,...,11.6,35,7300,4.600,0,0,0,1,0,0


In [68]:
cleaned_df = pd.concat([df_imputed, target_col], axis=1)

In [69]:
cleaned_df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48,80,1.020,1,0,1,1,0,0,121,...,44,7800,5.200,1,1,0,1,0,0,1
1,7,50,1.020,4,0,1,1,0,0,96,...,38,6000,5.334,0,0,0,1,0,0,1
2,62,80,1.010,2,3,1,1,0,0,423,...,31,7500,3.829,0,1,0,0,0,1,1
3,48,70,1.005,4,0,1,0,1,0,117,...,32,6700,3.900,1,0,0,0,1,1,1
4,51,80,1.010,2,0,1,1,0,0,106,...,35,7300,4.600,0,0,0,1,0,0,1


In [70]:
cleaned_df.isnull().sum()

,0
age,0
bp,0
sg,0
al,0
su,0
rbc,0
pc,0
pcc,0
ba,0
bgr,0


In [71]:
cleaned_df.to_csv("/content/drive/MyDrive/Colab Notebooks/CKD Project/cleaned_data_ckd.csv", index=False)

In [72]:
joblib.dump({"knn_imputer": knn_imputer,
             "imputer_scaler": scaler}, '/content/drive/MyDrive/Colab Notebooks/CKD Project/imputer_scaler.pkl')

['/content/drive/MyDrive/Colab Notebooks/CKD Project/imputer_scaler.pkl']